In [1]:
import pandas as pd
import numpy as np

# Display settings — shows all columns in output
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# Load the raw CSV — always read from data/raw/, never modify it
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# First look — always do this immediately after loading
print("Shape:", df.shape)       # rows × columns
print("\nFirst 5 rows:")
df.head()

Shape: (7043, 21)

First 5 rows:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# ============================================================
# DATA DICTIONARY — understand every column before touching it
# ============================================================

data_dict = {
    'customerID'       : 'Unique customer identifier (drop before modeling)',
    'gender'           : 'Male / Female',
    'SeniorCitizen'    : '1 if senior citizen, 0 otherwise (already numeric)',
    'Partner'          : 'Has a partner? Yes/No',
    'Dependents'       : 'Has dependents? Yes/No',
    'tenure'           : 'Months with the company (numeric)',
    'PhoneService'     : 'Has phone service? Yes/No',
    'MultipleLines'    : 'Multiple phone lines? Yes/No/No phone service',
    'InternetService'  : 'DSL / Fiber optic / No',
    'OnlineSecurity'   : 'Yes/No/No internet service',
    'OnlineBackup'     : 'Yes/No/No internet service',
    'DeviceProtection' : 'Yes/No/No internet service',
    'TechSupport'      : 'Yes/No/No internet service',
    'StreamingTV'      : 'Yes/No/No internet service',
    'StreamingMovies'  : 'Yes/No/No internet service',
    'Contract'         : 'Month-to-month / One year / Two year',
    'PaperlessBilling' : 'Yes/No',
    'PaymentMethod'    : 'Electronic check / Mailed check / Bank transfer / Credit card',
    'MonthlyCharges'   : 'Current monthly charge (numeric)',
    'TotalCharges'     : 'Total amount charged (WATCH OUT — stored as object, not float!)',
    'Churn'            : 'TARGET: Did customer churn? Yes/No'
}

for col, desc in data_dict.items():
    print(f"{col:<20} → {desc}")

customerID           → Unique customer identifier (drop before modeling)
gender               → Male / Female
SeniorCitizen        → 1 if senior citizen, 0 otherwise (already numeric)
Partner              → Has a partner? Yes/No
Dependents           → Has dependents? Yes/No
tenure               → Months with the company (numeric)
PhoneService         → Has phone service? Yes/No
MultipleLines        → Multiple phone lines? Yes/No/No phone service
InternetService      → DSL / Fiber optic / No
OnlineSecurity       → Yes/No/No internet service
OnlineBackup         → Yes/No/No internet service
DeviceProtection     → Yes/No/No internet service
TechSupport          → Yes/No/No internet service
StreamingTV          → Yes/No/No internet service
StreamingMovies      → Yes/No/No internet service
Contract             → Month-to-month / One year / Two year
PaperlessBilling     → Yes/No
PaymentMethod        → Electronic check / Mailed check / Bank transfer / Credit card
MonthlyCharges       → Curren

In [6]:
# ============================================================
# DATA QUALITY AUDIT — find all problems before fixing them
# ============================================================

print("=" * 55)
print("1. DATA TYPES")
print("=" * 55)
print(df.dtypes)

print("\n" + "=" * 55)
print("2. MISSING VALUES")
print("=" * 55)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

print("\n" + "=" * 55)
print("3. TARGET VARIABLE DISTRIBUTION")
print("=" * 55)
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100
print(pd.DataFrame({'Count': churn_counts, 'Percentage': churn_pct}))

print("\n" + "=" * 55)
print("4. UNIQUE VALUES IN CATEGORICAL COLUMNS")
print("=" * 55)
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    print(f"{col:<20} → {df[col].unique()}")

print("\n" + "=" * 55)
print("5. NUMERICAL COLUMNS SUMMARY")
print("=" * 55)
print(df.describe())

1. DATA TYPES
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

2. MISSING VALUES
Empty DataFrame
Columns: [Missing Count, Missing %]
Index: []

3. TARGET VARIABLE DISTRIBUTION
       Count  Percentage
Churn                   
No      5174   73.463013
Yes     1869   26.536987

4. UNIQUE VALUES IN CATEGORICAL COLUMNS
customerID           → ['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' ... '4801-JZAZL' '8361-LTMKD'
 '3186-AJIEK']
gender               

In [7]:
# ============================================================
# FINDING THE HIDDEN BUG — TotalCharges stored as object
# ============================================================

# TotalCharges SHOULD be numeric but is stored as a string.
# This happens in real datasets all the time.
# Let's find out why.

print("TotalCharges dtype:", df['TotalCharges'].dtype)
print("\nSample values:")
print(df['TotalCharges'].head(10).tolist())

# Try converting — errors='coerce' turns non-numeric values into NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# How many became NaN after conversion?
print("\nNaN count after conversion:", df['TotalCharges'].isnull().sum())

# Inspect those rows — what do they look like?
problem_rows = df[df['TotalCharges'].isnull()]
print("\nProblem rows:")
print(problem_rows[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']])

TotalCharges dtype: object

Sample values:
['29.85', '1889.5', '108.15', '1840.75', '151.65', '820.5', '1949.4', '301.9', '3046.05', '3487.95']

NaN count after conversion: 11

Problem rows:
      customerID  tenure  MonthlyCharges  TotalCharges Churn
488   4472-LVYGI       0           52.55           NaN    No
753   3115-CZMZD       0           20.25           NaN    No
936   5709-LVOEQ       0           80.85           NaN    No
1082  4367-NUYAO       0           25.75           NaN    No
1340  1371-DWPAZ       0           56.05           NaN    No
3331  7644-OMVMY       0           19.85           NaN    No
3826  3213-VVOLG       0           25.35           NaN    No
4380  2520-SGTTA       0           20.00           NaN    No
5218  2923-ARZLG       0           19.70           NaN    No
6670  4075-WKNIU       0           73.35           NaN    No
6754  2775-SEFEE       0           61.90           NaN    No


In [8]:
# ============================================================
# SQL ANALYSIS — use sqlite3 to query your DataFrame
# Industry skill: analysts query data in SQL, not just pandas
# ============================================================

import sqlite3

# Create an in-memory SQLite database and load the DataFrame
conn = sqlite3.connect(':memory:')
df.to_sql('customers', conn, index=False, if_exists='replace')

# Query 1: Churn rate by contract type
query1 = """
SELECT 
    Contract,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY Contract
ORDER BY churn_rate_pct DESC
"""
print("Churn Rate by Contract Type:")
print(pd.read_sql(query1, conn))

# Query 2: Average monthly charges for churned vs retained
query2 = """
SELECT 
    Churn,
    ROUND(AVG(MonthlyCharges), 2) AS avg_monthly_charge,
    ROUND(AVG(tenure), 1) AS avg_tenure_months,
    COUNT(*) AS customer_count
FROM customers
GROUP BY Churn
"""
print("\nChurned vs Retained — Avg Charges & Tenure:")
print(pd.read_sql(query2, conn))

# Query 3: Top payment methods among churners
query3 = """
SELECT 
    PaymentMethod,
    COUNT(*) AS churned_customers
FROM customers
WHERE Churn = 'Yes'
GROUP BY PaymentMethod
ORDER BY churned_customers DESC
"""
print("\nPayment Methods Among Churners:")
print(pd.read_sql(query3, conn))

conn.close()

Churn Rate by Contract Type:
         Contract  total_customers  churned  churn_rate_pct
0  Month-to-month             3875     1655           42.71
1        One year             1473      166           11.27
2        Two year             1695       48            2.83

Churned vs Retained — Avg Charges & Tenure:
  Churn  avg_monthly_charge  avg_tenure_months  customer_count
0    No               61.27               37.6            5174
1   Yes               74.44               18.0            1869

Payment Methods Among Churners:
               PaymentMethod  churned_customers
0           Electronic check               1071
1               Mailed check                308
2  Bank transfer (automatic)                258
3    Credit card (automatic)                232


In [9]:
# Save this version (TotalCharges fixed) as our starting point
# We always save to data/processed/ — never overwrite raw data

df.to_csv('../data/processed/telco_cleaned_v1.csv', index=False)
print("Saved to data/processed/telco_cleaned_v1.csv")
print(f"Shape: {df.shape}")

Saved to data/processed/telco_cleaned_v1.csv
Shape: (7043, 21)
